In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd


df = pd.read_csv(f"{path}/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
df

In [ ]:
df.isnull().sum()

In [ ]:
# Task 1: Write your code here:


num_cols = df.select_dtypes(include=["int64", "float64"]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())


df.isna().sum().sum()


In [ ]:
df.isnull().sum()

In [ ]:
# Task 2: Write your code here:
dup = df.duplicated().sum()
dup

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
df

In [ ]:
# Task 3: Write your code here:
#no needed

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler


target_col = "Target"
X = df.drop(columns=[target_col])
y = df[target_col]


num_cols = X.select_dtypes(include=["int64", "float64"]).columns

scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[num_cols] = scaler.fit_transform(X[num_cols])

X_scaled.shape
y.shape


In [ ]:
# Task 5: Write your code here:

counts = df["Target"].value_counts()
percent = df["Target"].value_counts(normalize=True) * 100

print("Target counts:\n", counts)
print("\nTarget %:\n", percent.round(2))

major = percent.max()
minor = percent.min()
ratio = major / minor

if (minor < 30) or (ratio >= 1.5):
    print("The target is IMBALANCED.")
else:
    print("The target is NOT imbalanced.")


In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1)
y = df["Target"]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# Task 2,3,4,5: Write your code here:
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier

In [ ]:
X_use = X_scaled if "X_scaled" in globals() else X
imbalanced = (y.value_counts(normalize=True).max() > 0.70)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) if imbalanced else KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
scores = []
for tr, te in cv.split(X_use, y if imbalanced else None):
    X_train, X_test = X_use.iloc[tr], X_use.iloc[te]
    y_train, y_test = y.iloc[tr], y.iloc[te]
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    scores.append(f1_score(y_test, pred, average="binary") if imbalanced else accuracy_score(y_test, pred))


In [ ]:
print(round(np.mean(scores), 4))

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
model.fit(X_use, y)


In [ ]:
importances = pd.Series(model.feature_importances_, index=X_use.columns).sort_values(ascending=False)
importances.head(15)


In [ ]:
importances.head(15).plot(kind="bar")
plt.title("Top 15 Feature Importances")
plt.xlabel("Features")
plt.ylabel("Importance")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

golden_feature = X_use.columns[model.feature_importances_.argmax()]
print(golden_feature)

In [ ]:
# Task Bonus: Write your code here:
X_golden = X_use[[golden_feature]]
X_golden.head()


In [ ]:
scores_golden = []
for tr, te in cv.split(X_golden, y if imbalanced else None):
    X_train, X_test = X_golden.iloc[tr], X_golden.iloc[te]
    y_train, y_test = y.iloc[tr], y.iloc[te]
    model_golden = RandomForestClassifier(random_state=42)
    model_golden.fit(X_train, y_train)
    pred = model_golden.predict(X_test)
    scores_golden.append(f1_score(y_test, pred, average="binary") if imbalanced else accuracy_score(y_test, pred))

print(round(np.mean(scores_golden), 4))


In [ ]:
print("Full model:", round(np.mean(scores), 4))
print("Golden feature:", round(np.mean(scores_golden), 4))